# API Fetch Development Notebook

In [1]:
import os
import sys
import requests
import pandas as pd

### Set Up Constants & API Environment

In [2]:
sys.path.insert(1,os.environ["PROJECT_ROOT"])
API_URL_ROOT = "https://financialmodelingprep.com/stable/"
api_key = os.environ["API_KEY"]

In [3]:
symbol = "NOPE"
response = requests.get(
    url = API_URL_ROOT+"quote",
    params = {"symbol":symbol,"apikey":api_key}
)
response.status_code

402

In [4]:
response

<Response [402]>

### Data Points and API

Symbol - quote<br>
Company Name - quote<br>
Reporting Currency - income statement<br>
Market Cap - quote<br>
Current Share Price - quote<br>
Latest Fiscal Year - income statement<br>
Latest Fiscal Year Revenue - income statement<br>
Latest Fiscal Year Net Income - income statement<br>
Profit Margin - income statement<br>
P/E Ratio - compute<br>
Earnings Per Share - compute<br>
Employees - none (drop)

In [5]:
def call_api(symbol,api,items_to_return):
    api = api.lower().strip()
    if api not in ["quote","income-statement"]:
        raise ValueError("'quote' and 'income-statement' are only APIs currently supported")
    else:
        url = os.path.join(API_URL_ROOT,api)
        print(f"Fetching {api.replace("-"," ")} from {url}")
        response = requests.get(
            url = url,
            params = {"symbol":symbol,"apikey":api_key}
        )
        response_dict = response.json()[0]
        print(f"Response received with length {len(response_dict)}")
        
    if items_to_return is None:
        return response_dict
    else:
        return {k:v for k,v in response_dict.items() if k in items_to_return}

### Build API Fetch and Dataframe Construction Functions

In [6]:
def get_quote_data(symbol):
    api = "quote"
    items_to_return = ["symbol","name","price","marketCap"]
    return call_api(symbol,api,items_to_return)

def get_income_statement_data(symbol):
    api = "income-statement"
    items_to_return = [
        "reportedCurrency",
        "fiscalYear",
        "revenue",
        "netIncome",
        "eps",
    ]
    return call_api(symbol,api,items_to_return)

def compute_fields_not_in_api(quote_data,income_statement_data):
    pe_ratio = float(quote_data["price"]) / float(income_statement_data["eps"])
    other_data = {"pe_ratio":pe_ratio}
    return other_data

def combine_data_into_dataframe(quote_data,income_statement_data,other_data):
    pretty_names = {
        "symbol":"Symbol",
        "name":"Company Name",
        "price":"Share Price",
        "marketCap":"Market Cap",
        "reportedCurrency":"Reported Currency",
        "fiscalYear":"Latest Completed Fiscal Year",
        "revenue":"Total Revenue",
        "netIncome":"Total Net Income",
        "eps":"Earnings Per Share",
        "pe_ratio":"Price to Earnings Ratio"
    }

    combined_dict = quote_data | income_statement_data | other_data
    df = pd.DataFrame(combined_dict,index = ["api_output"]).transpose().rename(pretty_names,axis = 0)
    
    return df


In [7]:
quote_data = get_quote_data("RIVN")
income_statement_data = get_income_statement_data("RIVN")
other_data = compute_fields_not_in_api(quote_data,income_statement_data)
df = combine_data_into_dataframe(quote_data,income_statement_data,other_data)
df

Fetching quote from https://financialmodelingprep.com/stable/quote
Response received with length 17
Fetching income statement from https://financialmodelingprep.com/stable/income-statement
Response received with length 39


,api_output
Symbol,RIVN
Company Name,"Rivian Automotive, Inc."
Share Price,15.036
Market Cap,18600694117
Reported Currency,USD
Latest Completed Fiscal Year,2025
Total Revenue,5387000000
Total Net Income,-3646000000
Earnings Per Share,-3.07
Price to Earnings Ratio,-4.89772


In [8]:
quote_data = {}
income_statement_data = {}
other_data = {}
df = combine_data_into_dataframe(quote_data,income_statement_data,other_data)

In [9]:
df

,api_output


### Test Functions from Production Script

In [10]:
import api_fetch

symbol = "TSLA"

In [11]:
quote_data = api_fetch.get_quote_data(symbol)
quote_data

Fetching quote from https://financialmodelingprep.com/stable/quote
Response received with length 17


{'symbol': 'TSLA',
 'name': 'Tesla, Inc.',
 'price': 402.4151,
 'marketCap': 1510035292085}

In [12]:
income_statement_data = api_fetch.get_income_statement_data(symbol)

Fetching income statement from https://financialmodelingprep.com/stable/income-statement
Response received with length 39


In [13]:
other_data = api_fetch.compute_fields_not_in_api(quote_data,income_statement_data)
other_data

{'pe_ratio': 341.02974576271185, 'net_profit_ratio': 0.04000970187815706}

In [14]:
df = api_fetch.combine_data_into_dataframe(quote_data,income_statement_data,other_data)
df

,api_output
Symbol,TSLA
Company Name,"Tesla, Inc."
Share Price,402.4151
Market Cap,1510035292085
Reported Currency,USD
Latest Completed Fiscal Year,2025
Total Revenue,94827000000
Total Net Income,3794000000
Earnings Per Share,1.18
Price to Earnings Ratio,341.029746


### Test Error Handling

In [15]:
api_fetch.get_quote_data("NOPE")

Fetching quote from https://financialmodelingprep.com/stable/quote
Retry attempt 1 of 2 for NOPE at https://financialmodelingprep.com/stable/quote...
Retry attempt 2 of 2 for NOPE at https://financialmodelingprep.com/stable/quote...


{}

In [16]:
api_fetch.get_income_statement_data("NOPE")

Fetching income statement from https://financialmodelingprep.com/stable/income-statement
Retry attempt 1 of 2 for NOPE at https://financialmodelingprep.com/stable/income-statement...
Retry attempt 2 of 2 for NOPE at https://financialmodelingprep.com/stable/income-statement...


{}

In [17]:
api_fetch.compute_fields_not_in_api({},{})

{}

In [18]:
api_fetch.combine_data_into_dataframe({},{},{})

,api_output
